# Lab 2 — CMC-13: Criação de Classificadores baseado em Dados

**Disciplina:** CMC-13 — Introdução a Ciência de Dados
**Variável alvo:** `Churn` (0 = cancelou o serviço · 1 = não cancelou)
**Dataset:** 440.832 amostras de treino · 64.374 amostras de teste · 12 features originais

---

## Estrutura deste notebook

| # | Seção | Conteúdo |
|---|---|---|
| 1 | **Preparação dos Dados** | Limpeza, codificação, EDA, investigação de problemas, correlações, exportação |
| 2 | **Modelos de Classificação** | KNN, Árvore de Decisão, SVM, MLP, Random Forest, AdaBoost, XGBoost |
| 3 | **Análise Comparativa** | Baselines, métricas e ranking final |

> **Metodologia comum a todos os modelos:** treino em X\_tr (80%), threshold calibrado
> via curva Precision-Recall em X\_val (20%), avaliação final exclusivamente no conjunto de teste.

---
## 1. Preparação dos Dados

Todo o pipeline de pré-processamento: desde o carregamento dos CSVs brutos até a exportação
dos dados escalados e prontos para os modelos. As etapas são:

1. Carregamento e diagnóstico inicial
2. Limpeza e codificação das variáveis
3. Análise exploratória (histogramas, KDE, distribuição por churn)
4. Investigação de quatro problemas detectados nos dados
5. Análise de correlação e seleção de features
6. Normalização e exportação em Parquet

### 1.1 Importações e Carregamento dos Dados Brutos

Carregamento das bibliotecas principais e leitura dos arquivos CSV de treino e teste.
O `sys.path` é configurado para tornar `lab2_utils` importável quando este notebook
é executado a partir da raiz do projeto.

In [ ]:
import sys
sys.path.insert(0, 'notebooks')  # torna lab2_utils importável a partir da raiz

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df_treino = pd.read_csv('material/lab2_cmc13_dados_treinamento.csv')
df_teste = pd.read_csv('material/lab2_cmc13_dados_teste.csv')

### 1.2 Diagnóstico Inicial

Verificação de dimensões, primeiras linhas, linhas completamente vazias e tipos de dados
em ambos os conjuntos. Revela diferenças entre treino (float64) e teste (int64)
que serão tratadas a seguir.

In [ ]:
print(f'Treino: {df_treino.shape[0]} linhas, {df_treino.shape[1]} colunas')
print(f'Teste:  {df_teste.shape[0]} linhas, {df_teste.shape[1]} colunas')
df_treino.head()

In [ ]:
# Linhas completamente vazias (todos os valores NaN)
linhas_vazias = df_treino[df_treino.isnull().all(axis=1)]
print(f"Linhas completamente vazias: {len(linhas_vazias)}")
print(f"Índices: {linhas_vazias.index.tolist()}")
linhas_vazias
# Remover linhas completamente vazias
df_treino = df_treino.dropna(how='all').reset_index(drop=True)
print(f"Treino após remoção: {df_treino.shape[0]} linhas")

In [ ]:
df_treino.info()
print("------------")
df_teste.info()

### 1.3 Limpeza e Codificação das Variáveis Categóricas

Transformações aplicadas **antes** da análise exploratória para que os gráficos
reflitam exatamente as features que entrarão nos modelos:

- **Remoção de `CustomerID`** — identificador sem valor preditivo
- **Conversão `float → int`** — artefato da linha NaN removida no treino
- **Label Encoding em `Gender`** — Male=0, Female=1
- **Ordinal Encoding em `Contract Length`** — Monthly=1, Quarterly=3, Annual=12
- **One-Hot Encoding em `Subscription Type`** — drop_first=True (Basic = referência)

## Ajustes Iniciais

Transformações aplicadas antes da análise exploratória para que os gráficos reflitam as features que serão efetivamente usadas nos modelos:

1. **Remover `CustomerID`** — identificador, sem valor preditivo
2. **Converter float → int** (treino) — artefato da linha NaN removida
3. **Label Encoding em `Gender`** — binária (Male=0, Female=1)
4. **Ordinal Encoding em `Contract Length`** — ordem natural de duração (Monthly=1, Quarterly=3, Annual=12)
5. **One-Hot Encoding em `Subscription Type`** — sem ordem natural entre categorias (drop_first=True)

In [ ]:
# 1. Remover CustomerID
df_treino = df_treino.drop(columns=['CustomerID'])
df_teste = df_teste.drop(columns=['CustomerID'])

# 2. Converter float → int (treino)
cols_float = df_treino.select_dtypes(include=[float]).columns
df_treino[cols_float] = df_treino[cols_float].astype(int)

# 3. Label Encoding — Gender (Male=0, Female=1)
df_treino['Gender'] = df_treino['Gender'].map({'Male': 0, 'Female': 1})
df_teste['Gender'] = df_teste['Gender'].map({'Male': 0, 'Female': 1})

# 4. Ordinal Encoding — Contract Length (Monthly=1, Quarterly=3, Annual=12)
df_treino['Contract Length'] = df_treino['Contract Length'].map({'Monthly': 1, 'Quarterly': 3, 'Annual': 12})
df_teste['Contract Length'] = df_teste['Contract Length'].map({'Monthly': 1, 'Quarterly': 3, 'Annual': 12})

# 5. One-Hot Encoding — Subscription Type (drop_first=True)
df_treino = pd.get_dummies(df_treino, columns=['Subscription Type'], drop_first=True, dtype=int)
df_teste = pd.get_dummies(df_teste, columns=['Subscription Type'], drop_first=True, dtype=int)

# Dicionário de labels para legibilidade nos gráficos
labels = {
    'Gender': {0: 'Male', 1: 'Female'},
    'Contract Length': {1: 'Monthly', 3: 'Quarterly', 12: 'Annual'}
}

print(f'Treino: {df_treino.shape} | Teste: {df_teste.shape}')
df_treino.head()

### 1.4 Análise Exploratória

Três camadas de visualização para entender o comportamento das features:

1. **Comparação estatística** treino vs teste (média, desvio, quartis, assimetria)
2. **Histogramas, KDE e boxplots** para as 7 features contínuas — treino e teste sobrepostos
3. **Countplots** para as features categóricas codificadas
4. **Distribuição por classe de churn** — identifica quais features separam melhor as classes

In [ ]:
# Comparação Treino vs Teste — todas as features (agora numéricas)
print("=" * 80)
print("COMPARAÇÃO TREINO vs TESTE - LADO A LADO")
print("=" * 80)

for col in df_treino.columns:
    print(f"\n{'─' * 80}")
    label_extra = ''
    if col in labels:
        label_extra = f"  ({', '.join(f'{k}={v}' for k, v in labels[col].items())})"
    print(f"  {col}{label_extra}")
    print(f"{'─' * 80}")
    
    stats = pd.DataFrame({
        'Treino': [
            df_treino[col].count(),
            df_treino[col].isnull().sum(),
            df_treino[col].nunique(),
            df_treino[col].mean(),
            df_treino[col].std(),
            df_treino[col].min(),
            df_treino[col].quantile(0.25),
            df_treino[col].median(),
            df_treino[col].quantile(0.75),
            df_treino[col].max(),
            df_treino[col].skew(),
            df_treino[col].kurtosis()
        ],
        'Teste': [
            df_teste[col].count(),
            df_teste[col].isnull().sum(),
            df_teste[col].nunique(),
            df_teste[col].mean(),
            df_teste[col].std(),
            df_teste[col].min(),
            df_teste[col].quantile(0.25),
            df_teste[col].median(),
            df_teste[col].quantile(0.75),
            df_teste[col].max(),
            df_teste[col].skew(),
            df_teste[col].kurtosis()
        ]
    }, index=['count', 'missing', 'únicos', 'mean', 'std', 'min', '25%', '50%', '75%', 'max', 'skew', 'kurtosis'])
    print(stats.to_string())

In [ ]:
# Features contínuas: Histograma + KDE + Boxplot
cols_continuas = ['Age', 'Tenure', 'Usage Frequency', 'Support Calls',
                  'Payment Delay', 'Total Spend', 'Last Interaction']

fig, axes = plt.subplots(len(cols_continuas), 3, figsize=(18, 4 * len(cols_continuas)))

for i, col in enumerate(cols_continuas):
    n_bins = df_treino[col].nunique()
    
    # Histograma
    axes[i, 0].hist(df_treino[col], bins=n_bins, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i, 0].set_title(f'{col} — Histograma')
    axes[i, 0].set_ylabel('Contagem')
    
    # KDE
    df_treino[col].plot.kde(ax=axes[i, 1], color='steelblue', label='Treino')
    df_teste[col].plot.kde(ax=axes[i, 1], color='coral', label='Teste')
    axes[i, 1].set_title(f'{col} — KDE (Treino vs Teste)')
    axes[i, 1].legend()
    
    # Boxplot
    axes[i, 2].boxplot(
        [df_treino[col], df_teste[col]],
        tick_labels=['Treino', 'Teste'], vert=True,
        patch_artist=True,
        boxprops=dict(facecolor='steelblue', alpha=0.5)
    )
    axes[i, 2].set_title(f'{col} — Boxplot')

plt.tight_layout()
plt.savefig('relatorio/imagens/1_prep_histogramas_numericas.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Features codificadas (ex-categóricas): Countplot com labels legíveis
cols_encoded = ['Gender', 'Contract Length', 'Subscription Type_Premium', 'Subscription Type_Standard']

fig, axes = plt.subplots(len(cols_encoded), 2, figsize=(14, 4 * len(cols_encoded)))

for i, col in enumerate(cols_encoded):
    for j, (df, nome, cor) in enumerate([(df_treino, 'Treino', 'steelblue'), (df_teste, 'Teste', 'coral')]):
        contagem = df[col].value_counts().sort_index()
        contagem.plot.bar(ax=axes[i, j], color=cor, edgecolor='white', alpha=0.8)
        axes[i, j].set_title(f'{col} — {nome}')
        axes[i, j].set_ylabel('Contagem')
        
        # Substituir rótulos numéricos pelos nomes originais
        if col in labels:
            tick_labels = [labels[col].get(int(v.get_text()), v.get_text()) for v in axes[i, j].get_xticklabels()]
            axes[i, j].set_xticklabels(tick_labels, rotation=0)
        else:
            axes[i, j].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('relatorio/imagens/1_prep_countplot_categoricas.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribuição por Churn — features contínuas
fig, axes = plt.subplots(len(cols_continuas), 1, figsize=(12, 4 * len(cols_continuas)))

for i, col in enumerate(cols_continuas):
    n_bins = df_treino[col].nunique()
    df_treino[df_treino['Churn'] == 0][col].plot.hist(
        ax=axes[i], bins=n_bins, alpha=0.6, color='steelblue', label='Não cancelou (0)', edgecolor='white'
    )
    df_treino[df_treino['Churn'] == 1][col].plot.hist(
        ax=axes[i], bins=n_bins, alpha=0.6, color='coral', label='Cancelou (1)', edgecolor='white'
    )
    axes[i].set_title(f'{col} — Distribuição por Churn')
    axes[i].set_ylabel('Contagem')
    axes[i].legend()

plt.tight_layout()
plt.savefig('relatorio/imagens/1_prep_distribuicao_por_churn.png', dpi=150, bbox_inches='tight')
plt.show()

# Distribuição por Churn — features codificadas
fig, axes = plt.subplots(1, len(cols_encoded), figsize=(6 * len(cols_encoded), 5))

for i, col in enumerate(cols_encoded):
    ct = pd.crosstab(df_treino[col], df_treino['Churn'], normalize='index')
    ct.plot.bar(ax=axes[i], color=['steelblue', 'coral'], edgecolor='white', alpha=0.8)
    axes[i].set_title(f'{col} — Proporção de Churn')
    axes[i].set_ylabel('Proporção')
    axes[i].legend(['Não cancelou (0)', 'Cancelou (1)'])
    
    if col in labels:
        tick_labels = [labels[col].get(int(v.get_text()), v.get_text()) for v in axes[i].get_xticklabels()]
        axes[i].set_xticklabels(tick_labels, rotation=0)
    else:
        axes[i].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('relatorio/imagens/1_prep_churn_por_categoricas.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.5 Investigação de Problemas

Quatro anomalias foram identificadas durante a EDA e investigadas formalmente antes
de qualquer decisão de tratamento.

### 5.1 — Granularidade: Total Spend

Os dados de treino originais tinham `Total Spend` com valores decimais (float64), enquanto o teste tinha valores inteiros. A conversão `float → int` no passo de ajustes iniciais já igualou a granularidade. Vamos confirmar que o tratamento foi efetivo.

In [ ]:
# Verificar granularidade atual (após conversão int)
print(f"Treino: {df_treino['Total Spend'].nunique()} valores únicos")
print(f"Teste:  {df_teste['Total Spend'].nunique()} valores únicos")

print("\nTreino (primeiros 10 valores ordenados):")
print(sorted(df_treino['Total Spend'].unique())[:10])
print("\nTeste (primeiros 10 valores ordenados):")
print(sorted(df_teste['Total Spend'].unique())[:10])

# Comparar dados originais (recarregar para mostrar a diferença)
df_original = pd.read_csv('material/lab2_cmc13_dados_treinamento.csv')
print(f"\n--- Antes do tratamento ---")
print(f"Treino original: {df_original['Total Spend'].nunique()} valores únicos (dtype: {df_original['Total Spend'].dtype})")
print(f"Primeiros 10 valores ordenados:")
print(sorted(df_original['Total Spend'].dropna().unique())[:10])

# Histograma sobreposto
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_treino['Total Spend'].hist(ax=axes[0], bins=50, alpha=0.7, color='steelblue', label='Treino')
df_teste['Total Spend'].hist(ax=axes[0], bins=50, alpha=0.7, color='coral', label='Teste')
axes[0].set_title('Total Spend — Distribuição (após tratamento)')
axes[0].legend()

# Zoom para confirmar granularidade igual
faixa_treino = df_treino['Total Spend'][(df_treino['Total Spend'] >= 500) & (df_treino['Total Spend'] <= 510)]
faixa_teste = df_teste['Total Spend'][(df_teste['Total Spend'] >= 500) & (df_teste['Total Spend'] <= 510)]
axes[1].hist(faixa_treino, bins=11, alpha=0.7, color='steelblue', label=f'Treino ({len(faixa_treino)} valores)')
axes[1].hist(faixa_teste, bins=11, alpha=0.7, color='coral', label=f'Teste ({len(faixa_teste)} valores)')
axes[1].set_title('Total Spend — Zoom 500-510')
axes[1].legend()
plt.tight_layout()
plt.savefig('relatorio/imagens/1_prep_granularidade_total_spend.png', dpi=150, bbox_inches='tight')
plt.show()

del df_original

**Resultado:** A conversão `float → int` nos ajustes iniciais já resolveu o problema de granularidade. Treino e teste agora têm os mesmos 901 valores únicos (inteiros de 100 a 1000). Nenhum tratamento adicional necessário.

### 5.2 — Separação perfeita: Contract Length

Nos gráficos de proporção de churn por categoria, `Contract Length = Monthly` apresentou taxa de churn próxima a 100% no treino. Se esse padrão não se mantiver no teste, há risco de overfitting nessa feature.

In [ ]:
# Taxa de churn por Contract Length — treino vs teste
churn_treino = df_treino.groupby('Contract Length')['Churn'].mean()
churn_teste = df_teste.groupby('Contract Length')['Churn'].mean()

comparacao = pd.DataFrame({
    'Treino (% churn)': (churn_treino * 100).round(1),
    'Teste (% churn)': (churn_teste * 100).round(1),
    'Treino (n)': df_treino.groupby('Contract Length')['Churn'].count(),
    'Teste (n)': df_teste.groupby('Contract Length')['Churn'].count()
})
comparacao.index = comparacao.index.map(labels['Contract Length'])
print(comparacao.to_string())

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
churn_treino.plot.bar(ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Treino — Taxa de Churn por Contract Length')
axes[0].set_ylabel('Taxa de Churn')
axes[0].set_xticklabels([labels['Contract Length'][int(float(v.get_text()))] for v in axes[0].get_xticklabels()], rotation=0)
axes[0].set_ylim(0, 1.1)

churn_teste.plot.bar(ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Teste — Taxa de Churn por Contract Length')
axes[1].set_ylabel('Taxa de Churn')
axes[1].set_xticklabels([labels['Contract Length'][int(float(v.get_text()))] for v in axes[1].get_xticklabels()], rotation=0)
axes[1].set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig('relatorio/imagens/1_prep_separacao_perfeita_contract.png', dpi=150, bbox_inches='tight')
plt.show()

**Resultado:** Separação perfeita confirmada — Monthly tem **100% churn no treino** (87.104 amostras) mas apenas **51.6% no teste** (22.130 amostras). Quarterly e Annual se comportam de forma consistente entre treino (~46%) e teste (~44-46%).

O padrão do treino não generaliza para o teste. Qualquer modelo vai aprender "Monthly = churn garantido" e errar metade das predições no teste.

> **DECISÃO FINAL (Melhoria 1 — Pareto):** `Contract Length` será **removida** antes da exportação do parquet.
> A separação perfeita no treino (Monthly=100% churn) que não se repete no teste (Monthly=51.6% churn)
> é a causa raiz do colapso preditivo observado: recall≈1, acurácia≈0.5, Kappa≈0.
> Experimentos anteriores sem essa feature resultaram em desempenho mais baixo, mas esse cenário
> foi testado COM os modelos ainda sofrendo do colapso — com o threshold calibrado (Melhoria 2),
> a ausência do sinal enviesado deve produzir modelos genuinamente discriminativos.

### 5.3 — Teste de Data Leakage

Suspeita: `Support Calls` e `Contract Length` podem ser **consequência** do churn, não causa. Se uma árvore de decisão rasa (max_depth=3) usando apenas essas 2 features atingir acurácia > 95%, o modelo está "lendo a resposta" em vez de aprender padrões reais.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Teste com apenas Support Calls + Contract Length
X_leak = df_treino[['Support Calls', 'Contract Length']]
y_leak = df_treino['Churn']
X_leak_teste = df_teste[['Support Calls', 'Contract Length']]
y_leak_teste = df_teste['Churn']

arvore = DecisionTreeClassifier(max_depth=3, random_state=42)
arvore.fit(X_leak, y_leak)

acc_treino = accuracy_score(y_leak, arvore.predict(X_leak))
acc_teste = accuracy_score(y_leak_teste, arvore.predict(X_leak_teste))

print(f"Acurácia com apenas 2 features (Support Calls + Contract Length):")
print(f"  Treino: {acc_treino:.4f}")
print(f"  Teste:  {acc_teste:.4f}")
print(f"\nSe acurácia > 95%, há forte suspeita de data leakage.\n")
print("Relatório no teste:")
print(classification_report(y_leak_teste, arvore.predict(X_leak_teste)))

# Comparar com modelo usando TODAS as features
X_all = df_treino.drop(columns=['Churn'])
X_all_teste = df_teste.drop(columns=['Churn'])

arvore_all = DecisionTreeClassifier(max_depth=3, random_state=42)
arvore_all.fit(X_all, df_treino['Churn'])

acc_all_treino = accuracy_score(df_treino['Churn'], arvore_all.predict(X_all))
acc_all_teste = accuracy_score(df_teste['Churn'], arvore_all.predict(X_all_teste))

print(f"\nAcurácia com TODAS as features:")
print(f"  Treino: {acc_all_treino:.4f}")
print(f"  Teste:  {acc_all_teste:.4f}")

**Resultado:** A acurácia com apenas 2 features ficou em **84.3% no treino** e **61.7% no teste** — abaixo do limiar de 95%, então **não há data leakage clássico**. Support Calls e Contract Length não "entregam a resposta" sozinhas.

No entanto, a queda treino→teste é preocupante: **23 pontos** com 2 features e **34 pontos** com todas. Adicionar mais features piorou o teste (de 61.7% para 58.5%), indicando que as features extras ajudam o modelo a decorar o treino em vez de generalizar. Esse comportamento reforça a decisão de remover Contract Length (principal responsável pelo overfitting). Support Calls será mantida — não apresenta separação perfeita, e os modelos mostrarão se o data shift afeta o desempenho.

### 5.4 — Desbalanceamento: Contract Length

`Monthly` representa ~20% das amostras no treino vs ~33% no teste. Combinado com o comportamento extremo (≈100% churn), essa diferença de proporção pode afetar a generalização dos modelos.

In [ ]:
# Contagem e proporção por Contract Length — treino vs teste
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, df, nome, cor in [(axes[0], df_treino, 'Treino', 'steelblue'), (axes[1], df_teste, 'Teste', 'coral')]:
    contagem = df['Contract Length'].value_counts().sort_index()
    bars = contagem.plot.bar(ax=ax, color=cor, edgecolor='white', alpha=0.8)
    ax.set_title(f'{nome} — Amostras por Contract Length')
    ax.set_ylabel('Contagem')
    ax.set_xticklabels([labels['Contract Length'][int(float(v.get_text()))] for v in ax.get_xticklabels()], rotation=0)

    # Adicionar % em cima de cada barra
    total = len(df)
    for bar, val in zip(bars.patches, contagem):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + total*0.01,
                f'{val/total*100:.1f}%', ha='center', fontsize=11)

plt.tight_layout()
plt.savefig('relatorio/imagens/1_prep_desbalanceamento_contract.png', dpi=150, bbox_inches='tight')
plt.show()

**Resultado:** A proporção de Monthly no treino (~20%) é muito diferente do teste (~33%), enquanto Quarterly e Annual são ~40% no treino e ~33% no teste.

> **DECISÃO FINAL:** `Contract Length` será removida (ver seção 5.2). A combinação de separação
> perfeita no treino + shift de distribuição no teste justifica a remoção. A célula abaixo executa a remoção.

#### Aplicação da Decisão: Remoção de `Contract Length`

Com base nas evidências das seções 5.2 e 5.4 — separação perfeita no treino
(Monthly = 100% churn) que não se repete no teste (Monthly ≈ 51,6%) —,
`Contract Length` é removida de ambos os conjuntos. Esta foi a melhoria de maior
impacto no projeto.

In [ ]:
df_treino = df_treino.drop(columns=['Contract Length'])
df_teste = df_teste.drop(columns=['Contract Length'])

print(f'Treino: {df_treino.shape} | Teste: {df_teste.shape}')
print(f'Features restantes: {[c for c in df_treino.columns if c != "Churn"]}')

### 1.6 Análise de Correlação

Análise multidimensional de correlação entre features e com a variável alvo:

- **Pearson, Spearman e Kendall** — detecta relações lineares e monotônicas
- **Point-Biserial** — correlação de cada feature numérica com `Churn`
- **Análise de `Subscription Type`** — reconstrução da categoria *Basic* oculta pelo `drop_first`
- **Comparação Pearson vs Spearman** — verifica se há relações não-lineares relevantes

## Análise de Correlação

Correlação calculada com todas as features disponíveis após os ajustes iniciais (encoding, remoção de CustomerID).

In [ ]:
# Correlação — Pearson, Spearman, Kendall
metodos = ['pearson', 'spearman', 'kendall']

fig, axes = plt.subplots(3, 1, figsize=(12, 36))

for i, metodo in enumerate(metodos):
    matriz = df_treino.corr(method=metodo)
    sns.heatmap(
        matriz, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
        vmin=-1, vmax=1, ax=axes[i], square=True,
        linewidths=0.5, cbar_kws={'shrink': 0.8}
    )
    axes[i].set_title(f'Correlação {metodo.capitalize()}', fontsize=14)

plt.tight_layout()
plt.savefig('relatorio/imagens/1_prep_correlacao_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Point-Biserial: correlação de cada feature com Churn
from scipy.stats import pointbiserialr

print("=== Correlação Point-Biserial com Churn ===\n")

cols_features = [c for c in df_treino.columns if c != 'Churn']

resultados = {}
for col in cols_features:
    corr, pvalue = pointbiserialr(df_treino['Churn'], df_treino[col])
    resultados[col] = {'correlação': corr, 'p-value': pvalue}

pb = pd.DataFrame(resultados).T.sort_values('correlação', key=abs, ascending=False)
pb['correlação'] = pb['correlação'].map('{:.4f}'.format)
pb['p-value'] = pb['p-value'].map('{:.2e}'.format)
print(pb.to_string())

# Visualização
fig, ax = plt.subplots(figsize=(10, 6))
vals = pd.DataFrame(resultados).T.sort_values('correlação', key=abs, ascending=True)
colors = ['coral' if v > 0 else 'steelblue' for v in vals['correlação']]
ax.barh(vals.index, vals['correlação'], color=colors, edgecolor='white')
ax.set_title('Point-Biserial: Correlação com Churn')
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set_xlabel('Correlação')
plt.tight_layout()
plt.savefig('relatorio/imagens/1_prep_point_biserial.png', dpi=150, bbox_inches='tight')
plt.show()

### Seleção de Features pela Correlação

Dois critérios para considerar a remoção de uma feature:
1. **Baixa correlação com Churn** — feature com pouco poder preditivo
2. **Alta correlação entre features** (multicolinearidade) — features redundantes que podem prejudicar alguns modelos (ex: regressão logística, SVM)

In [ ]:
# === ANÁLISE COMPLETA DE CORRELAÇÃO ===

cols_features = [c for c in df_treino.columns if c != 'Churn']

# 1. Correlação de cada feature com Churn
corr_churn = {}
for col in cols_features:
    corr, pvalue = pointbiserialr(df_treino['Churn'], df_treino[col])
    corr_churn[col] = round(corr, 4)

# 2. Matriz de correlação entre features (sem Churn)
corr_features = df_treino[cols_features].corr()

# 3. Para cada feature: maior correlação com outra feature
max_corr_par = {}
for col in cols_features:
    outros = corr_features[col].drop(col).abs().sort_values(ascending=False)
    top = outros.index[0]
    max_corr_par[col] = {
        'par': top,
        'valor': round(corr_features.loc[col, top], 4)
    }

# Tabela resumo
resumo = pd.DataFrame({
    'Corr. com Churn': corr_churn,
    '|Corr. Churn|': {k: abs(v) for k, v in corr_churn.items()},
    'Par mais correlacionado': {k: v['par'] for k, v in max_corr_par.items()},
    'Corr. com par': {k: v['valor'] for k, v in max_corr_par.items()},
    '|Corr. par|': {k: abs(v['valor']) for k, v in max_corr_par.items()},
}).sort_values('|Corr. Churn|', ascending=False)

print("=== RESUMO: Correlação com Churn + Maior correlação entre features ===\n")
print(resumo.to_string())

# 4. Todos os pares de features com correlação significativa
print("\n\n=== TODOS os pares de features (|corr| > 0.05) ===\n")
pares = []
for i in range(len(cols_features)):
    for j in range(i+1, len(cols_features)):
        val = corr_features.iloc[i, j]
        if abs(val) > 0.05:
            pares.append({
                'Feature 1': cols_features[i],
                'Feature 2': cols_features[j],
                'Correlação': round(val, 4),
                '|Corr|': round(abs(val), 4)
            })

df_pares = pd.DataFrame(pares).sort_values('|Corr|', ascending=False)
print(df_pares.to_string(index=False))

In [ ]:
# Visualização: Heatmap apenas entre features (sem Churn) + barras de correlação com Churn
fig, axes = plt.subplots(1, 2, figsize=(20, 8), gridspec_kw={'width_ratios': [2, 1]})

# Heatmap feature-feature
sns.heatmap(
    corr_features, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
    vmin=-1, vmax=1, ax=axes[0], square=True,
    linewidths=0.5, cbar_kws={'shrink': 0.8}
)
axes[0].set_title('Correlação entre Features (sem Churn)', fontsize=13)

# Barras de correlação com Churn
ordem = sorted(corr_churn.items(), key=lambda x: abs(x[1]))
nomes = [x[0] for x in ordem]
valores = [x[1] for x in ordem]
cores = ['coral' if v > 0 else 'steelblue' for v in valores]

axes[1].barh(nomes, valores, color=cores, edgecolor='white')
axes[1].axvline(x=0, color='black', linewidth=0.5)
axes[1].axvline(x=0.1, color='gray', linewidth=0.5, linestyle='--', alpha=0.5)
axes[1].axvline(x=-0.1, color='gray', linewidth=0.5, linestyle='--', alpha=0.5)
axes[1].axvline(x=0.3, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
axes[1].axvline(x=-0.3, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
axes[1].set_title('Correlação com Churn', fontsize=13)
axes[1].set_xlabel('Point-Biserial')

plt.tight_layout()
plt.savefig('relatorio/imagens/1_prep_correlacao_completa.png', dpi=150, bbox_inches='tight')
plt.show()

### Problema do One-Hot com `drop_first=True`

O `drop_first=True` suprimiu a categoria **Basic**, que virou a referência implícita (Premium=0, Standard=0). Isso causa dois problemas na análise:
1. A correlação de Basic com Churn fica **invisível** — não sabemos se Basic é relevante
2. Premium e Standard ficam com correlação artificial de -0.51 entre si (são mutuamente exclusivas)

Para analisar corretamente, reconstruímos temporariamente as 3 categorias:

In [ ]:
# Reconstruir Basic e analisar as 3 categorias separadamente
df_treino['Subscription Type_Basic'] = ((df_treino['Subscription Type_Premium'] == 0) & 
                                         (df_treino['Subscription Type_Standard'] == 0)).astype(int)

print("=== Correlação de cada categoria do Subscription Type com Churn ===\n")
for cat in ['Subscription Type_Basic', 'Subscription Type_Premium', 'Subscription Type_Standard']:
    corr, pvalue = pointbiserialr(df_treino['Churn'], df_treino[cat])
    proporcao = df_treino[cat].mean() * 100
    churn_rate = df_treino[df_treino[cat] == 1]['Churn'].mean() * 100
    print(f"{cat:>30}: corr={corr:+.4f}  p={pvalue:.2e}  |  {proporcao:.1f}% das amostras  |  churn={churn_rate:.1f}%")

# Taxa de churn geral para comparar
print(f"\n{'Taxa de churn geral':>30}: {df_treino['Churn'].mean()*100:.1f}%")

# Visualização: proporção de churn por categoria
fig, ax = plt.subplots(figsize=(8, 5))
cats = ['Basic', 'Premium', 'Standard']
churn_rates = [
    df_treino[df_treino['Subscription Type_Basic'] == 1]['Churn'].mean(),
    df_treino[df_treino['Subscription Type_Premium'] == 1]['Churn'].mean(),
    df_treino[df_treino['Subscription Type_Standard'] == 1]['Churn'].mean()
]
bars = ax.bar(cats, churn_rates, color=['steelblue', 'coral', 'seagreen'], edgecolor='white', alpha=0.8)
ax.axhline(y=df_treino['Churn'].mean(), color='black', linestyle='--', label=f'Média geral ({df_treino["Churn"].mean()*100:.1f}%)')
for bar, rate in zip(bars, churn_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{rate*100:.1f}%', ha='center', fontsize=12)
ax.set_ylabel('Taxa de Churn')
ax.set_title('Taxa de Churn por Subscription Type (todas as categorias)')
ax.legend()
plt.tight_layout()
plt.savefig('relatorio/imagens/1_prep_churn_subscription_completo.png', dpi=150, bbox_inches='tight')
plt.show()

# Remover coluna temporária
df_treino = df_treino.drop(columns=['Subscription Type_Basic'])

### Correlações não-lineares

A correlação de Pearson (e Point-Biserial) mede apenas relações **lineares**. Uma feature pode ter correlação linear fraca mas relação **monotônica** (capturada por Spearman/Kendall) ou padrão **não-monotônico** (ex: formato U) que só aparece nos gráficos de distribuição por Churn.

Comparamos os 3 métodos para as 4 features candidatas a remoção e referenciamos os gráficos de distribuição por Churn já gerados na seção de Análise Exploratória.

In [ ]:
# Comparação Pearson vs Spearman vs Kendall para as 4 features fracas
from scipy.stats import spearmanr, kendalltau

candidatas = ['Tenure', 'Usage Frequency', 'Subscription Type_Premium', 'Subscription Type_Standard']

print("=== Pearson vs Spearman vs Kendall — Features com correlação linear fraca ===\n")
print(f"{'Feature':>30} | {'Pearson':>10} | {'Spearman':>10} | {'Kendall':>10} | Mudou?")
print("-" * 85)

for col in candidatas:
    pearson = df_treino[col].corr(df_treino['Churn'])
    spearman, _ = spearmanr(df_treino[col], df_treino['Churn'])
    kendall, _ = kendalltau(df_treino[col], df_treino['Churn'])
    
    mudou = 'SIM' if abs(spearman) > abs(pearson) * 1.5 else 'não'
    print(f"{col:>30} | {pearson:>+10.4f} | {spearman:>+10.4f} | {kendall:>+10.4f} | {mudou}")

# Mesma análise para TODAS as features (contexto)
print("\n\n=== Mesma comparação para TODAS as features ===\n")
print(f"{'Feature':>30} | {'Pearson':>10} | {'Spearman':>10} | {'Kendall':>10}")
print("-" * 75)

for col in [c for c in df_treino.columns if c != 'Churn']:
    pearson = df_treino[col].corr(df_treino['Churn'])
    spearman, _ = spearmanr(df_treino[col], df_treino['Churn'])
    kendall, _ = kendalltau(df_treino[col], df_treino['Churn'])
    print(f"{col:>30} | {pearson:>+10.4f} | {spearman:>+10.4f} | {kendall:>+10.4f}")

**Conclusões da análise de correlação:**

**Subscription Type:**
- Com `drop_first=True`, Basic ficou oculta. A análise com as 3 categorias revelou taxas de churn quase idênticas: Basic=58.2%, Premium=55.9%, Standard=56.1% (média geral=56.7%). Subscription Type não discrimina churn.

**Correlações não-lineares (Pearson vs Spearman vs Kendall):**
- Nenhuma das 4 features fracas apresentou mudança significativa entre os métodos — a relação é genuinamente fraca, não há padrão monotônico oculto.
- As features fortes (Support Calls, Total Spend, Payment Delay) também mantêm valores consistentes entre métodos, confirmando que a relação com Churn é predominantemente linear.

> **DECISÃO SUSPENSA (dados experimentais):** A análise de correlação indicava remover
> Tenure (corr=0.05), Usage Frequency (corr=0.05), Subscription Type_Premium (corr=0.01)
> e Subscription Type_Standard (corr=0.01) por baixa correlação com Churn.
>
> No entanto, ao treinar os modelos apenas com as 6 features restantes (Age, Gender,
> Support Calls, Payment Delay, Total Spend, Last Interaction), todos os classificadores
> (DecisionTree, MLP, RandomForest) apresentaram desempenho quase aleatório
> (acurácia ~52%, Kappa ~0.09). Correlação linear fraca não significa ausência de
> poder preditivo — os modelos podem capturar interações não-lineares entre features.
>
> **Todas as features serão mantidas.** O impacto será avaliado via MLflow comparando
> runs com 6 vs 11 features.

In [ ]:
# SUSPENSA — Remoção de features revertida (ver markdown acima)
# A correlação linear fraca não implica ausência de poder preditivo.
# cols_remover = ['Tenure', 'Usage Frequency', 'Subscription Type_Premium', 'Subscription Type_Standard']
# df_treino = df_treino.drop(columns=cols_remover)
# df_teste = df_teste.drop(columns=cols_remover)

print(f'Treino: {df_treino.shape} | Teste: {df_teste.shape}')
print(f'TODAS as features mantidas: {[c for c in df_treino.columns if c != "Churn"]}')

### 1.7 Visão Geral dos Dados Finais

Após todas as transformações, resumo completo das 10 features que entrarão nos modelos,
incluindo distribuição treino vs teste (KDE) e distribuição por classe de churn para
cada feature — base para o relato no relatório.

## Dados Finais

Visão geral das features que serão enviadas para os modelos, após todos os tratamentos e remoções.

In [ ]:
# Resumo dos dados finais
print(f"Treino: {df_treino.shape[0]} amostras, {df_treino.shape[1] - 1} features + target")
print(f"Teste:  {df_teste.shape[0]} amostras, {df_teste.shape[1] - 1} features + target")
print(f"\nFeatures: {[c for c in df_treino.columns if c != 'Churn']}")
print(f"Target: Churn")
print(f"\nDistribuição do target:")
print(f"  Treino — Churn=1: {df_treino['Churn'].mean()*100:.1f}% | Churn=0: {(1-df_treino['Churn'].mean())*100:.1f}%")
print(f"  Teste  — Churn=1: {df_teste['Churn'].mean()*100:.1f}% | Churn=0: {(1-df_teste['Churn'].mean())*100:.1f}%")
df_treino.describe().round(2)

In [ ]:
# KDE Treino vs Teste — features finais
cols_finais = [c for c in df_treino.columns if c != 'Churn']
n_cols = 4
n_rows = -(-len(cols_finais) // n_cols)  # ceil division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(cols_finais):
    df_treino[col].plot.kde(ax=axes[i], color='steelblue', label='Treino')
    df_teste[col].plot.kde(ax=axes[i], color='coral', label='Teste')
    axes[i].set_title(col, fontsize=12)
    axes[i].legend(fontsize=9)

for j in range(len(cols_finais), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuição Treino vs Teste — Features Finais', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('relatorio/imagens/1_prep_kde_features_finais.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribuição por Churn — features finais
n_cols = 4
n_rows = -(-len(cols_finais) // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(cols_finais):
    n_bins = min(df_treino[col].nunique(), 30)
    df_treino[df_treino['Churn'] == 0][col].plot.hist(
        ax=axes[i], bins=n_bins, alpha=0.6, color='steelblue', label='Não cancelou (0)', edgecolor='white'
    )
    df_treino[df_treino['Churn'] == 1][col].plot.hist(
        ax=axes[i], bins=n_bins, alpha=0.6, color='coral', label='Cancelou (1)', edgecolor='white'
    )
    axes[i].set_title(col, fontsize=12)
    axes[i].legend(fontsize=8)

for j in range(len(cols_finais), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuição por Churn — Features Finais', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('relatorio/imagens/1_prep_churn_features_finais.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.8 Normalização e Exportação dos Dados Processados

`StandardScaler` ajustado **exclusivamente no treino** e aplicado ao teste (sem
vazamento de informação). Os dados escalados são exportados em formato Parquet para
reutilização em todos os notebooks de modelagem. A verificação final confirma que
as médias do treino escalado são ≈ 0 para todas as features.

## Exportação dos Dados Processados

Exporta os DataFrames de treino e teste em formato **Parquet** para que os notebooks de modelagem (2A, 2B, 2C) possam importar os dados já preparados.

**Importante:**
- Os dados são exportados **já escalonados** com `StandardScaler` (fit no treino, transform no teste).
- `Contract Length` foi **removida** antes da exportação (ver seção 5.2 — separação perfeita + shift).
- Escalonamento não afeta modelos baseados em árvore (splits são invariantes a transformações monotônicas), então todos os notebooks usam os mesmos dados.
- Para regenerar os dados, basta re-executar este notebook inteiro.

In [ ]:
import os
from datetime import datetime
from sklearn.preprocessing import StandardScaler

target = 'Churn'

X_train = df_treino.drop(columns=[target])
y_train = df_treino[[target]]
X_test = df_teste.drop(columns=[target])
y_test = df_teste[[target]]

# Escalonamento centralizado — fit no treino, transform no teste
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

output_dir = 'notebooks/dados_processados'
os.makedirs(output_dir, exist_ok=True)

X_train_scaled.to_parquet(f'{output_dir}/X_train.parquet', index=False)
y_train.to_parquet(f'{output_dir}/y_train.parquet', index=False)
X_test_scaled.to_parquet(f'{output_dir}/X_test.parquet', index=False)
y_test.to_parquet(f'{output_dir}/y_test.parquet', index=False)

print(f'Exportado em: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Diretório: {os.path.abspath(output_dir)}')
print(f'\nX_train: {X_train_scaled.shape} | y_train: {y_train.shape}')
print(f'X_test:  {X_test_scaled.shape} | y_test:  {y_test.shape}')
print(f'\nFeatures: {list(X_train.columns)}')
print(f'Scaler: StandardScaler (fit no treino)')
print(f'\nVerificação (médias do treino escalado):')
print(X_train_scaled.mean().round(6).to_string())

---
## 2. Modelos de Classificação

Sete classificadores são treinados e avaliados. Todos seguem a **mesma metodologia**:

1. O conjunto de treino é dividido em **X\_tr** (80%) e **X\_val** (20%), estratificado.
2. Cada modelo é treinado em **X\_tr** — o teste jamais é visto nessa etapa.
3. O **threshold de decisão** é calibrado em **X\_val** via curva Precision-Recall,
   encontrando o limiar que maximiza o F1 na validação. Isso evita o limiar fixo 0,5,
   que se mostrou inadequado dado o desbalanceamento residual entre treino e teste.
4. A **avaliação final** usa apenas o conjunto de teste (64.374 amostras).

### 2.0 Configuração — Carregamento e Split de Validação

Carregamento dos dados processados em Parquet e criação do split X\_tr / X\_val
dentro do treino. Essas variáveis são **compartilhadas por todos os modelos**
nas seções seguintes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from lab2_utils import (
    carregar_dados, avaliar_modelo, logar_mlflow, iniciar_run,
    calibrar_threshold, predizer_com_threshold,
)

X_train, y_train, X_test, y_test = carregar_dados(data_dir='notebooks/dados_processados')

# Split estratificado — X_tr para treino, X_val para calibração de threshold
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

print(f"X_tr:    {X_tr.shape}   (80% do treino — usado para fit)")
print(f"X_val:   {X_val.shape}  (20% do treino — usado para calibrar threshold)")
print(f"X_test:  {X_test.shape}  (teste final — nunca visto durante o desenvolvimento)")

### 2.1 Modelos Tradicionais

Três classificadores clássicos de machine learning, cada um com paradigma diferente:

| Modelo | Paradigma |
|---|---|
| **KNN** | Similaridade — classifica pela maioria das *k* vizinhas mais próximas |
| **Árvore de Decisão** | Regras — aprende divisões binárias hierárquicas nas features |
| **SVM** | Margem — encontra o hiperplano que maximiza a distância entre classes |

#### 2.1.1 KNN — K-Nearest Neighbors

O KNN classifica cada amostra pela votação das *k* vizinhas mais próximas no espaço
de features. Com `weights='distance'`, vizinhas mais próximas têm peso maior na
votação, melhorando o desempenho em fronteiras difusas.

**Hiperparâmetros:**
- `n_neighbors = 11` — valor ímpar para evitar empates; suaviza a fronteira de decisão
- `weights = 'distance'` — peso proporcional à proximidade
- Threshold calibrado via `predict_proba` + curva Precision-Recall em X\_val

In [ ]:
# ─── KNN ────────────────────────────────────────────────────────
# Treina em X_tr (80% do treino), calibra threshold via predict_proba
# no X_val (20%), prediz no X_test com threshold ótimo.
from sklearn.neighbors import KNeighborsClassifier

params = {
    'modelo': 'KNN',
    'n_neighbors': 11,
    'weights': 'distance',
    'metric': 'minkowski',
    'scaler': 'StandardScaler (centralizado)',
}

with iniciar_run("KNN", notebook="2A", params=params):
    model = KNeighborsClassifier(
        n_neighbors=11,
        weights='distance',
        n_jobs=-1,
    )
    model.fit(X_tr, y_tr)

    threshold = calibrar_threshold(model, X_val, y_val)
    y_pred = predizer_com_threshold(model, X_test, threshold)
    params['threshold'] = round(threshold, 4)
    print(f'Threshold calibrado: {threshold:.4f}')

    metricas = avaliar_modelo('KNN', y_test, y_pred)
    logar_mlflow(metricas, params)

#### 2.1.2 Árvore de Decisão

Classificador que aprende regras binárias ("se feature X > threshold, então...") de forma
hierárquica. Restrições de profundidade e de amostras mínimas por folha controlam
a complexidade e evitam memorização do treino.

**Hiperparâmetros:**
- `max_depth = 5` — limita a profundidade máxima para evitar overfitting
- `min_samples_split = 50` — exige pelo menos 50 amostras para dividir um nó
- `min_samples_leaf = 20` — cada folha precisa de no mínimo 20 amostras
- `criterion = 'gini'` — impureza de Gini como critério de divisão
- Threshold calibrado via `predict_proba` + curva PR em X\_val

In [ ]:
# ─── Árvore de Decisão ──────────────────────────────────────────
# Threshold calibrado via predict_proba substitui o ajuste por class_weight.
from sklearn.tree import DecisionTreeClassifier

params = {
    'modelo': 'DecisionTree',
    'max_depth': 5,
    'min_samples_split': 50,
    'min_samples_leaf': 20,
    'criterion': 'gini',
    'scaler': 'StandardScaler (centralizado)',
}

with iniciar_run("DecisionTree", notebook="2A", params=params):
    model = DecisionTreeClassifier(
        max_depth=5,
        min_samples_split=50,
        min_samples_leaf=20,
        criterion='gini',
        random_state=42,
    )
    model.fit(X_tr, y_tr)

    threshold = calibrar_threshold(model, X_val, y_val)
    y_pred = predizer_com_threshold(model, X_test, threshold)
    params['threshold'] = round(threshold, 4)
    print(f'Threshold calibrado: {threshold:.4f}')

    metricas = avaliar_modelo('DecisionTree', y_test, y_pred)
    logar_mlflow(metricas, params)

#### 2.1.3 SVM — Support Vector Machine

O SVM linear encontra o hiperplano que separa as classes com a maior margem possível.
Como o `LinearSVC` não fornece probabilidades, ele é encapsulado em
`CalibratedClassifierCV` (3 folds) para habilitar `predict_proba` — requisito
para a calibração do threshold via curva PR.

**Hiperparâmetros:**
- `C = 0.1` — regularização forte; penaliza menos erros em favor de margem maior
- `class_weight = 'balanced'` — compensa o desbalanceamento treino/teste
- `CalibratedClassifierCV(cv=3)` — adiciona `predict_proba` via calibração Platt
- Threshold calibrado via curva PR em X\_val

In [ ]:
# ─── SVM ────────────────────────────────────────────────────────
# CalibratedClassifierCV adiciona predict_proba ao LinearSVC,
# permitindo calibração de threshold via curva PR.
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

params = {
    'modelo': 'SVM',
    'tipo': 'CalibratedLinearSVC',
    'C': 0.1,
    'class_weight': 'balanced',
    'max_iter': 2000,
    'scaler': 'StandardScaler (centralizado)',
}

with iniciar_run("SVM", notebook="2A", params=params):
    base_svm = LinearSVC(
        C=0.1,
        class_weight='balanced',
        max_iter=2000,
        random_state=42,
    )
    model = CalibratedClassifierCV(base_svm, cv=3)
    model.fit(X_tr, y_tr)

    threshold = calibrar_threshold(model, X_val, y_val)
    y_pred = predizer_com_threshold(model, X_test, threshold)
    params['threshold'] = round(threshold, 4)
    print(f'Threshold calibrado: {threshold:.4f}')

    metricas = avaliar_modelo('SVM', y_test, y_pred)
    logar_mlflow(metricas, params)

### 2.2 Rede Neural — MLP (MultiLayer Perceptron)

O MLP aprende representações hierárquicas das features por meio de múltiplas camadas
de neurônios com funções de ativação não lineares. Cada camada transforma a
representação da camada anterior, permitindo capturar padrões que modelos lineares
não conseguem.

**Arquitetura:** entrada (10 features) → camada oculta 1 (64 neurônios, ReLU)
→ camada oculta 2 (32 neurônios, ReLU) → saída (1 neurônio, sigmoid implícito)

**Hiperparâmetros:**
- `alpha = 0.01` — regularização L2 que penaliza pesos grandes (reduz overfitting)
- `solver = 'adam'` — otimizador adaptativo eficiente para datasets grandes
- `learning_rate_init = 0.001` — taxa de aprendizado inicial
- `early_stopping = True` — interrompe o treino quando validação interna para melhorar
- `max_iter = 100` — limite máximo de épocas
- Threshold calibrado via `predict_proba` + curva PR em X\_val

In [ ]:
# ─── MLP ──────────────────────────────────────────────────────
# Treina em X_tr (80%), calibra threshold via predict_proba no X_val (20%).
# early_stopping usa internamente 10% do X_tr como validação de parada.
from sklearn.neural_network import MLPClassifier

hidden = (64, 32)
params = {
    'modelo': 'MLP',
    'hidden_layer_sizes': str(hidden),
    'learning_rate_init': 0.001,
    'alpha': 0.01,
    'max_iter': 100,
    'activation': 'relu',
    'solver': 'adam',
    'early_stopping': True,
    'scaler': 'StandardScaler (centralizado)',
}

with iniciar_run("MLP", notebook="2B", params=params):
    model = MLPClassifier(
        hidden_layer_sizes=hidden,
        learning_rate_init=0.001,
        alpha=0.01,
        max_iter=100,
        activation='relu',
        solver='adam',
        early_stopping=True,
        validation_fraction=0.1,
        random_state=42,
    )
    model.fit(X_tr, y_tr)

    threshold = calibrar_threshold(model, X_val, y_val)
    y_pred = predizer_com_threshold(model, X_test, threshold)
    params['threshold'] = round(threshold, 4)
    print(f'Threshold calibrado: {threshold:.4f}')

    metricas = avaliar_modelo('MLP', y_test, y_pred)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(model.loss_curve_, label='Treino')
    if hasattr(model, 'validation_scores_'):
        ax.plot(model.validation_scores_, label='Validação')
    ax.set_title('Loss por Época — MLP')
    ax.set_xlabel('Época')
    ax.set_ylabel('Loss / Score')
    ax.legend()
    plt.tight_layout()
    loss_path = 'relatorio/imagens/2b_rn_loss_por_epoca.png'
    plt.savefig(loss_path, dpi=150, bbox_inches='tight')
    plt.show()

    logar_mlflow(metricas, params, artefatos=[loss_path])

### 2.3 Modelos de Comitê (Ensembles)

Os modelos de comitê combinam múltiplos classificadores para obter previsões mais
robustas do que qualquer modelo individual. Duas estratégias foram utilizadas:

- **Bagging (Random Forest):** treina árvores independentes em subamostras, combina por votação
- **Boosting (AdaBoost e XGBoost):** treina estimadores sequencialmente, cada um focado
  nos erros do anterior

#### 2.3.1 Random Forest

Floresta de 200 árvores independentes, cada uma treinada em uma subamostra aleatória
dos dados e das features. A combinação por votação majoritária reduz a variância
individual de cada árvore.

**Hiperparâmetros:**
- `n_estimators = 200` — 200 árvores na floresta
- `max_depth = 10` — profundidade máxima por árvore (regularização)
- `min_samples_split = 20` e `min_samples_leaf = 10` — folhas mais robustas
- Threshold calibrado via `predict_proba` em X\_val
- Feature Importance plotada para interpretabilidade

In [ ]:
# ─── Random Forest ──────────────────────────────────────────────
# Threshold calibrado via predict_proba substitui class_weight='balanced'.
from sklearn.ensemble import RandomForestClassifier

params = {
    'modelo': 'RandomForest',
    'n_estimators': 200,
    'max_depth': 10,
    'min_samples_split': 20,
    'min_samples_leaf': 10,
    'scaler': 'nenhum (dados já escalados)',
}

with iniciar_run("RandomForest", notebook="2C", params=params):
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=20,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_tr, y_tr)

    threshold = calibrar_threshold(model, X_val, y_val)
    y_pred = predizer_com_threshold(model, X_test, threshold)
    params['threshold'] = round(threshold, 4)
    print(f'Threshold calibrado: {threshold:.4f}')

    metricas = avaliar_modelo('RandomForest', y_test, y_pred)

    fig, ax = plt.subplots(figsize=(10, 6))
    importances = pd.Series(model.feature_importances_, index=X_tr.columns).sort_values()
    importances.plot.barh(ax=ax, color='steelblue', edgecolor='white')
    ax.set_title('Feature Importance — Random Forest')
    plt.tight_layout()
    fi_path = 'relatorio/imagens/2c_com_feature_importance_rf.png'
    plt.savefig(fi_path, dpi=150, bbox_inches='tight')
    plt.show()

    logar_mlflow(metricas, params, artefatos=[fi_path])

#### 2.3.2 AdaBoost

O AdaBoost treina estimadores fracos sequencialmente, aumentando o peso das amostras
mal classificadas pelo estimador anterior. O resultado final é uma soma ponderada
de todos os estimadores.

**Hiperparâmetros:**
- `n_estimators = 200` — 200 estimadores fracos (stumps de decisão)
- `learning_rate = 0.05` — contribuição conservadora de cada estimador;
  convergência mais lenta e estável, reduzindo o risco de overfitting
- Threshold calibrado via `predict_proba` em X\_val

In [ ]:
# ─── AdaBoost ───────────────────────────────────────────────────
# Threshold calibrado via predict_proba no split de validação.
from sklearn.ensemble import AdaBoostClassifier

params = {
    'modelo': 'AdaBoost',
    'n_estimators': 200,
    'learning_rate': 0.05,
    'scaler': 'nenhum (dados já escalados)',
}

with iniciar_run("AdaBoost", notebook="2C", params=params):
    model = AdaBoostClassifier(
        n_estimators=200,
        learning_rate=0.05,
        random_state=42,
    )
    model.fit(X_tr, y_tr)

    threshold = calibrar_threshold(model, X_val, y_val)
    y_pred = predizer_com_threshold(model, X_test, threshold)
    params['threshold'] = round(threshold, 4)
    print(f'Threshold calibrado: {threshold:.4f}')

    metricas = avaliar_modelo('AdaBoost', y_test, y_pred)
    logar_mlflow(metricas, params)

#### 2.3.3 XGBoost

Implementação otimizada de Gradient Boosting com regularização nativa.
Cada estimador é treinado para corrigir os resíduos do ensemble anterior.

**Hiperparâmetros:**
- `n_estimators = 200` — 200 estimadores no ensemble
- `max_depth = 4` — árvores mais rasas para melhor generalização
- `learning_rate = 0.05` — contribuição por estimador
- `reg_alpha = 0.1` — regularização L1 (esparsidade)
- `reg_lambda = 1.0` — regularização L2 (suavidade)
- `eval_metric = 'logloss'` — métrica de avaliação interna
- Threshold calibrado via `predict_proba` em X\_val

In [ ]:
# ─── XGBoost ────────────────────────────────────────────────────
# Threshold calibrado via predict_proba substitui scale_pos_weight.
from xgboost import XGBClassifier

params = {
    'modelo': 'XGBoost',
    'n_estimators': 200,
    'max_depth': 4,
    'learning_rate': 0.05,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'scaler': 'nenhum (dados já escalados)',
}

with iniciar_run("XGBoost", notebook="2C", params=params):
    model = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',
    )
    model.fit(X_tr, y_tr)

    threshold = calibrar_threshold(model, X_val, y_val)
    y_pred = predizer_com_threshold(model, X_test, threshold)
    params['threshold'] = round(threshold, 4)
    print(f'Threshold calibrado: {threshold:.4f}')

    metricas = avaliar_modelo('XGBoost', y_test, y_pred)
    logar_mlflow(metricas, params)

---
## 3. Análise Comparativa dos Modelos

Comparação do desempenho de todos os 7 classificadores treinados, incluindo dois
baselines triviais (*DummyClassifiers*) como referência do acaso.

**Métricas utilizadas:**

| Métrica | Descrição |
|---|---|
| **Balanced Accuracy** | (Recall₀ + Recall₁) / 2 — **métrica principal de ranking** |
| Acurácia | Proporção de predições corretas |
| Precision | TP / (TP + FP) — qualidade dos positivos preditos |
| Recall | TP / (TP + FN) — cobertura dos positivos reais |
| F1-Score | Média harmônica de Precision e Recall |
| Kappa de Cohen | Concordância descontando o acaso |

> **Por que Balanced Accuracy e não F1?** Com recall ≈ 1,0 e precision ≈ 0,5,
> o F1 fica inflado mesmo para modelos sem poder discriminativo real. Um *Dummy*
> que prevê sempre churn=1 obtém F1 = 0,643 — quase igual ao MLP (0,663).
> A Balanced Accuracy não é afetada: o mesmo Dummy obtém 0,500 (puro acaso).

### 3.1 Baselines — DummyClassifier

Antes de comparar os modelos reais entre si, estabelecemos a **linha do acaso**
com dois classificadores triviais que não aprendem padrão algum dos dados:

- **`most_frequent`** — prevê sempre a classe mais frequente no treino (churn=1, ~56,7%)
- **`stratified`** — prevê aleatoriamente respeitando a distribuição do treino

Todo modelo real só tem utilidade prática se superar esses baselines em
Balanced Accuracy (> 0,500).

In [ ]:
from lab2_utils import buscar_melhores_runs
from sklearn.dummy import DummyClassifier

# X_train, y_train, X_test, y_test já estão definidos na seção 2

In [ ]:
from sklearn.dummy import DummyClassifier

dummies = [
    ('Dummy_MostFrequent', DummyClassifier(strategy='most_frequent', random_state=42)),
    ('Dummy_Stratified',   DummyClassifier(strategy='stratified',    random_state=42)),
]

for nome, dummy in dummies:
    params = {'modelo': nome, 'strategy': dummy.strategy}
    with iniciar_run(nome, notebook="3", params=params):
        dummy.fit(X_train, y_train)
        y_pred = dummy.predict(X_test)
        metricas = avaliar_modelo(nome, y_test, y_pred)
        logar_mlflow(metricas, params)

### 3.2 Carregamento das Métricas via MLflow

A função `buscar_melhores_runs()` recupera o run **mais recente** de cada modelo
do experimento `Lab2-Churn` registrado no MLflow. Os resultados incluem todos os
modelos treinados nas seções 2.1, 2.2 e 2.3, além dos *Dummies* acima.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from lab2_utils import (
    buscar_melhores_runs, carregar_dados,
    avaliar_modelo, logar_mlflow, iniciar_run,
)

X_train, y_train, X_test, y_test = carregar_dados(data_dir='notebooks/dados_processados')

### 3.3 Comparação Visual das Métricas

Gráfico de barras com todas as métricas para cada modelo, ordenados por Balanced
Accuracy decrescente. A linha vermelha tracejada marca Balanced Accuracy = 0,500
(nível do acaso — referência dos *Dummies*).

In [ ]:
metricas_cols = ['Balanced_Acc', 'Acuracia', 'Precision', 'Recall', 'F1-Score', 'Kappa']

df_plot = df_metricas[metricas_cols].sort_values('Balanced_Acc', ascending=False)

fig, ax = plt.subplots(figsize=(16, 6))
df_plot.plot.bar(ax=ax, edgecolor='white', alpha=0.85)
ax.set_title('Comparacao de Desempenho dos Modelos (ordenado por Balanced Accuracy)', fontsize=14)
ax.set_ylabel('Valor da Metrica')
ax.set_ylim(0, 1.05)
ax.axhline(0.5, color='red', linestyle='--', linewidth=1, alpha=0.6, label='Linha do acaso (0.5)')
ax.legend(loc='lower right')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('relatorio/imagens/3_comp_barplot_metricas.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4 Ranking Final dos Modelos

Ranking completo por Balanced Accuracy (métrica principal) e por Kappa de Cohen.
Os *Dummies* aparecem ao final como referência — qualquer modelo com Balanced
Accuracy > 0,500 ou Kappa > 0 supera o acaso.

**Resultado esperado:** SVM lidera com Balanced Accuracy ≈ 0,604 e Kappa ≈ 0,200.

In [ ]:
print('=== Ranking por Balanced Accuracy (metrica principal) ===')
print(df_metricas.sort_values('Balanced_Acc', ascending=False)[metricas_cols].to_string())

print('\n=== Ranking por Kappa ===')
print(df_metricas.sort_values('Kappa', ascending=False)[metricas_cols].to_string())

print('\n=== Dummies (linha do acaso) ===')
dummies_idx = [m for m in df_metricas.index if 'Dummy' in str(m)]
if dummies_idx:
    print(df_metricas.loc[dummies_idx, metricas_cols].to_string())